# Notebook 2: Model Training

This notebook demonstrates:
- LSTM training with regularization (dropout, L2, early stopping)
- Training curves (loss and accuracy over epochs)
- Hyperparameter search (grid search over 6 configurations)
- XGBoost and Random Forest training
- Ensemble combination via meta-learner
- Inference timing measurements

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from src.data.collector import StockDataCollector
from src.data.preprocessor import DataPreprocessor
from src.data.dataset import StockSequenceDataset, create_dataloaders
from src.features.technical import TechnicalIndicators
from src.models.lstm_model import LSTMPredictor
from src.models.tree_models import TreeEnsemble
from src.models.ensemble import EnsembleModel
from src.training.trainer import LSTMTrainer
from src.training.hyperopt import HyperparameterSearch, SMALL_GRID
from src.evaluation.metrics import TradingMetrics

plt.style.use('seaborn-v0_8-darkgrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
SEQ_LEN = 20
TICKER = 'AAPL'

## 1. Data Setup

In [ ]:
# Load data and compute features
collector = StockDataCollector(tickers=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA'])
prices = collector.download_prices(start='2015-01-01', end='2024-01-01')

preprocessor = DataPreprocessor()
ohlcv, preproc_stats = preprocessor.preprocess_prices(prices, TICKER)
features_df = TechnicalIndicators.compute_all(ohlcv)

# Load daily sentiment scores produced by notebook 1
import os as _os
_sent_path = "../data/daily_sentiment.csv"
if _os.path.exists(_sent_path):
    import pandas as _pd
    _sent = _pd.read_csv(_sent_path, parse_dates=["date"])
    _sent_t = _sent[_sent["ticker"] == TICKER].set_index("date")["sentiment_score"]
    features_df["sentiment_score"] = features_df.index.map(_sent_t).fillna(0.0)
    _n = features_df["sentiment_score"].abs().gt(0).sum()
    print(f"Loaded sentiment for {TICKER}: {_n} scored days")
else:
    features_df["sentiment_score"] = 0.0
    print("No sentiment file — run notebook 1 first. Using zeros.")

combined = features_df.join(ohlcv[['LogReturn']], how='inner').dropna()
returns = combined['LogReturn'].values
feature_cols = [c for c in combined.columns if c != 'LogReturn']
feature_array = combined[feature_cols].values

n = len(feature_array)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Fit scaler on training data only (prevent data leakage)
preprocessor.fit_scaler(feature_array[:train_end])
features_scaled = preprocessor.transform(feature_array)

print(f'Features: {features_scaled.shape[1]}')
print(f'Train: {train_end}, Val: {val_end - train_end}, Test: {n - val_end}')

## 2. PyTorch DataLoaders

In [ ]:
train_ds = StockSequenceDataset(features_scaled[:train_end], returns[:train_end], SEQ_LEN)
val_ds   = StockSequenceDataset(features_scaled[train_end:val_end], returns[train_end:val_end], SEQ_LEN)
test_ds  = StockSequenceDataset(features_scaled[val_end:], returns[val_end:], SEQ_LEN)

train_loader, val_loader, test_loader = create_dataloaders(
    train_ds, val_ds, test_ds, batch_size=64
)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')
print(f'Train class distribution: {train_ds.class_distribution}')
print(f'Input shape per batch: {next(iter(train_loader))[0].shape}')

# Compute class weights for imbalanced dataset
labels = (returns[:train_end] > 0).astype(int)
class_weights_np = preprocessor.compute_class_weights(labels)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)
print(f'Class weights: {class_weights}')

## 3. LSTM Training with Regularization

In [ ]:
# Train the best LSTM configuration
model = LSTMPredictor(
    input_size=features_scaled.shape[1],
    hidden_size=128,
    num_layers=2,
    dropout=0.3,  # Regularization technique 1: dropout
)
print(model)

trainer = LSTMTrainer(
    model=model,
    learning_rate=1e-3,
    weight_decay=1e-4,  # Regularization technique 2: L2 weight decay
    patience=15,         # Regularization technique 3: early stopping
    class_weights=class_weights,
    device=DEVICE,
)

history = trainer.fit(train_loader, val_loader, epochs=100, verbose=True)
print(f'\nTraining complete. Epochs trained: {len(history["train_loss"])}')

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(epochs, history['val_loss'], label='Val Loss', color='#FF5722')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# Accuracy curves
axes[1].plot(epochs, history['train_acc'], label='Train Acc', color='#2196F3')
axes[1].plot(epochs, history['val_acc'], label='Val Acc', color='#FF5722')
axes[1].axhline(0.5, color='black', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()

# Learning rate schedule
axes[2].plot(epochs, history['lr'], color='#4CAF50')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule (ReduceLROnPlateau)')
axes[2].set_yscale('log')

plt.suptitle('LSTM Training Curves (hidden=128, layers=2, dropout=0.3)', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best val_acc: {max(history["val_acc"]):.4f}')
print(f'Best val_loss: {min(history["val_loss"]):.4f}')

## 5. Hyperparameter Search (Grid Search)

In [ ]:
# Grid search over 6 configurations (requirement: >= 3)
search = HyperparameterSearch(
    input_size=features_scaled.shape[1],
    train_loader=train_loader,
    val_loader=val_loader,
    grid=SMALL_GRID,
    epochs_per_trial=30,
    patience=8,
    device=DEVICE,
    class_weights=class_weights,
)

best_config, all_results = search.run()
print('\n=== Hyperparameter Search Results ===')
print(search.summary_table())
print(f'\nBest configuration: {best_config}')

In [ ]:
# Visualize hyperparameter search results
results_df = pd.DataFrame(all_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val accuracy by hidden size
for layers in results_df['num_layers'].unique():
    subset = results_df[results_df['num_layers'] == layers]
    axes[0].scatter(subset['hidden_size'], subset['val_acc'],
                   s=100, label=f'{layers} layer(s)')
axes[0].set_xlabel('Hidden Size')
axes[0].set_ylabel('Val Accuracy')
axes[0].set_title('Val Accuracy vs Hidden Size')
axes[0].legend()

# Summary bar chart
labels_hp = [f"h={r['hidden_size']},d={r['dropout']}" for r in all_results]
val_accs = [r['val_acc'] for r in all_results]
colors_hp = ['#4CAF50' if r == max(val_accs) else '#2196F3' for r in val_accs]
axes[1].barh(labels_hp, val_accs, color=colors_hp)
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].set_xlabel('Validation Accuracy')
axes[1].set_title('All Configurations Compared')
axes[1].legend()

plt.suptitle('Hyperparameter Search: 6 Configurations Compared', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/hyperparameter_search.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tree Model Training

In [ ]:
labels_all = (returns > 0).astype(int)
X_train = features_scaled[:train_end]
y_train = labels_all[:train_end]
X_val   = features_scaled[train_end:val_end]
y_val   = labels_all[train_end:val_end]

tree_models = TreeEnsemble()
tree_metrics = tree_models.fit(X_train, y_train, X_val, y_val)
print('Tree model metrics:')
for k, v in tree_metrics.items():
    print(f'  {k}: {v:.4f}')

# Feature importance
importance = tree_models.get_feature_importance(feature_cols)
top_10 = list(importance.items())[:10]

fig, ax = plt.subplots(figsize=(10, 5))
feat_names, importances = zip(*top_10)
ax.barh(list(feat_names), list(importances), color='#2196F3')
ax.set_xlabel('Importance')
ax.set_title('XGBoost Feature Importances (Top 10)')
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Ensemble Training

In [ ]:
# Get validation predictions from all models
val_lstm_proba = trainer.predict_proba(val_loader)
val_xgb_proba, val_rf_proba = tree_models.predict_proba(X_val[:len(val_lstm_proba)])
val_labels = labels_all[train_end:val_end][:len(val_lstm_proba)]

ensemble = EnsembleModel()
ensemble_metrics = ensemble.fit_meta_learner(
    val_lstm_proba, val_xgb_proba, val_rf_proba, val_labels
)
print('Ensemble metrics:', ensemble_metrics)

# Compare individual models vs ensemble on validation
from sklearn.metrics import accuracy_score, f1_score

comparison = {
    'LSTM': accuracy_score(val_labels, val_lstm_proba.argmax(axis=1)),
    'XGBoost': accuracy_score(val_labels, val_xgb_proba.argmax(axis=1)),
    'Random Forest': accuracy_score(val_labels, val_rf_proba.argmax(axis=1)),
    'Ensemble': ensemble_metrics['ensemble_val_acc'],
}

print('\nModel Comparison on Validation Set:')
for m, acc in comparison.items():
    print(f'  {m}: {acc:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
colors_cmp = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
bars = ax.bar(comparison.keys(), comparison.values(), color=colors_cmp)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Model Comparison: Individual vs Ensemble')
ax.legend()
ax.set_ylim(0.45, 0.65)
for bar, acc in zip(bars, comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.002, f'{acc:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../docs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Inference Timing

In [ ]:
# Measure inference time for each model
import time

sample_X = torch.tensor(test_ds.X[:64], dtype=torch.float32).to(DEVICE)
model.eval()

# LSTM timing
times_lstm = []
for _ in range(20):
    with torch.no_grad():
        t0 = time.perf_counter()
        _ = model(sample_X)
        times_lstm.append(time.perf_counter() - t0)

lstm_timing = TradingMetrics.measure_inference_time(
    lambda x: model(torch.tensor(x, dtype=torch.float32).to(DEVICE)),
    test_ds.X[:64]
)

print('Inference Timing Results:')
print(f'  LSTM mean latency:  {np.mean(times_lstm)*1000:.3f} ms')
print(f'  LSTM throughput:    {64/np.mean(times_lstm):.1f} samples/sec')
print(f'  Full timing report: {lstm_timing}')